# Welter Game Calculations

The Welter game is a game of perfect information in the are of Combinatorial Game Theory (CGT) analyzed by Welter in the 1950s, 
with an exposition by John Conway in his classic book On Numbers And Games.  This discussion follows Conway's notation.  
An equivalent version of the game was studied by Soto, phrased as a game on number partitions.  I am studying the mathematics of this game with two collaborators. This file containts a collection of Python functions and classes associated with the game, including the ability to actually play the game, using a console interface.

The Game Rules: The game board is a strip of spaces numbered $0, 1, 2, \ldots, k$ for some positive integer $k$.  
The game begins with some number of coins being placed in these spaces, at most one coin to a space, with the rest of the spaces being empty.
For example, if $k = 6$, the game could begin in the position 
$$ {\large \unicode{x24EA}} \ {\large \unicode{x2460}} \ 2 \ 3 \ {\large \unicode{x2463}} \ 5 \ {\large \unicode{x2465}},$$
with spots 0, 1, 3, and 6 occupied, while spots 2, 4, and 5 start out empty.
The two players alternate play till one player cannot move - the player who cannot move loses (i.e. the standard
notion of "normal play").  On a turn a player must choose some coin to move to a lower numbered empty spot (crucially, 
you CAN jump over coins).  For example in the above start position, the first player could move the 
coin at spot 6 down to spots 2, 4, or 5.

The following is an example of complete game play, beginning with the above starting position.

- Player 1: $ {\large \unicode{x24EA}} \ {\large \unicode{x2460}} \ 2 \ 3 \ {\large \unicode{x2463}} \ 
{\large \unicode{x2464}} \ 6 $ (by moving the coin at spot 6 to spot 5)

- Player 2: $ {\large \unicode{x24EA}} \ {\large \unicode{x2460}} \ {\large \unicode{x2461}} \ 3 \ {\large \unicode{x2463}} \ 5 \ 6 $ (by moving the coin at spot 5 to spot 2)

- Player 1:  $ {\large \unicode{x24EA}} \ {\large \unicode{x2460}} \ {\large \unicode{x2461}} \ {\large \unicode{x2462}} \ 4 \ 5 \ 6 $ (by moving the coin at spot 4 to spot 3)

At this point, player 2 has no legal move, so player 1 is the winner.

The analysis of this game (originally by Welter) fully solves this game in classic CGT style.
Without giving all the details of CGT, note the following useful definition:  The player who's turn it is is called
the Next player, while player who just played is called the Previous player.  A position in this game is either
a N-position, meaning the Next player can guarantee a win, or a P-position, meaning the Previous player can guarantee a win.  For example, the 
initial position above is an N-position because if the Next player plays smart (as indicated by the series of moves above), then that player can 
guarantee a win.  Welter develops a rather involved way to calculate the Grundy value (a non-negative integer) of any game position, where as is standard in CGT, the value 0 corresponds to a P-position, and a postive value corresponds to a N-position.
Thus if it is a players turn (i.e. they are in the role of the Next player) and the current value is positive, they can choose a game
position whose value is 0, so their opponent will be forced to give them a game position with a positive value.  The player
can iterate in this manner until they win.

This file goes on to write functions and classes to carry out the various calculations needed for playing the game.
<b>Skip towards the bottom of the file to just play the game!<b>



In [114]:
# All python packages used

import itertools as it # used below in the Even Alternation Theory
import random as rd    # used below in the functionality to play the game

## Basic Utility Functions

The following simple functions are defined in Conway's development of the Welter proof

In [80]:
def modPowTwo(a, b):
    '''
    Input: 2 distint integers
    Output: The exponent of the largest power of 2 difference
    Example: modPowTwo(3, 11) = 3 since 11 - 3 = 8 = 2^3, the exponent is 3.
    '''

    if a == b or a != int(a) or b != int(b):
        raise Exception("modPowTwo must receive distinct integers")
    
    diff = abs(b - a)
    k = 0
    while True:
        if diff % 2**k != 0:
            break

        k = k + 1

    k = k - 1 # since went to the first place of disagreement, go back one

    return k

def test_modPowTwo():
    print("modPowTwo(2, 3) = 0?", modPowTwo(2, 3))
    print("modPowTwo(20, 18) = 1?", modPowTwo(20, 18))
    print("modPowTwo(11, 3) = 3?", modPowTwo(11, 3))
    print("modPowTwo(3, 11) = 3?", modPowTwo(3, 11))
    print("modPowTwo(7, -3) = 1?", modPowTwo(7, -3))

    # Should raise exceptions if uncommented
    '''
    print("Exception?", modPowTwo(7.8, 7))    
    print("Exception?", modPowTwo(7, 7))
    '''

In [81]:
test_modPowTwo()

modPowTwo(2, 3) = 0? 0
modPowTwo(20, 18) = 1? 1
modPowTwo(11, 3) = 3? 3
modPowTwo(3, 11) = 3? 3
modPowTwo(7, -3) = 1? 1


In [82]:
def mate(a, b):
    '''
    Input: Two integers
    Output: Returns the mating function value of a and b,
            which is set to -1 if a = b; 
            otherwise, it equals 2^(n + 1) - 1, where n = modPowTwo(a, b) 
    Example: mate(3, 11) = 2^(3 + 1) - 1 = 15
    '''
    if a != int(a) or b != int(b):
        raise Exception("mating funcion must receive integers")
    
    if a == b:
        return -1
    else:
        return 2**(modPowTwo(a, b) + 1) - 1

def test_mate():
    print("True? - mate(3, 11) = 2^4 - 1 = 15?", mate(3, 11) )
    print("True? - mate(3, 4) = 2^1 - 1 = 1?", mate(3, 4) )
    print("mate(4, 4) = -1?", mate(4, 4) )

In [83]:
test_mate()

True? - mate(3, 11) = 2^4 - 1 = 15? 15
True? - mate(3, 4) = 2^1 - 1 = 1? 1
mate(4, 4) = -1? -1


In [84]:
'''
This extended XOR is based on calculations from our notes, extending XOR to allow positive or negative integers
'''

def xorE(a, b):
    'Extended XOR - i.e. can have negative a or b'
    # First make sure a <= b by swapping values if a > b
    if a > b:
        a, b = b, a
    
    if a >= 0 and b >= 0:
        return a^b             # ordinary XOR
    elif a <= -2 and b >=0:
        return -1 - (-a-1)^b
    else:
        return (-a-1)^(-b-1)

def test_xorE():
    print("xorE(5, 1) = 4?", xorE(5, 1) )
    print("xorE(0, 9) = 9?", xorE(0, 9) )
    print("xorE(-1, 2) = -3?", xorE(-1, 2) )
    print("xorE(-3, 7) = -6?", xorE(-3, 7) )
    print("xorE(7, -3) = -6?", xorE(7, -3) )
    print("xorE(-2, -3) = 3?", xorE(-2, -3) )
    print("xorE(2, -4) = -2?", xorE(2, -4) )
    print("xorE(-4, 2) = -2?", xorE(-4, 2) )

    

In [85]:
test_xorE()

xorE(5, 1) = 4? 4
xorE(0, 9) = 9? 9
xorE(-1, 2) = -3? -3
xorE(-3, 7) = -6? -6
xorE(7, -3) = -6? -6
xorE(-2, -3) = 3? 3
xorE(2, -4) = -2? -2
xorE(-4, 2) = -2? -2


In [86]:
'The classic Nim Sum is just the repeated xor applied to the numbers; here extended to negtive integers'

def nim(L):
    '''
    Input: A list of integers
    Output: The nim sum of the integers in the list
    '''
    val = 0
    for x in L:
        val = xorE(val, x)

    return val

def test_nim():
    print("nim([3,2,3]) = 2?", nim([3,2,3]) )
    print("nim([3,2,3,1]) = 3?", nim([3,2,3,1]) )

In [87]:
test_nim()

nim([3,2,3]) = 2? 2
nim([3,2,3,1]) = 3? 3


### Welter Function

The Welter function is the key function; it is used to calculate the Grundy value of a game position.  Given
a list of integers $a_1, \ldots, a_k$, 

$$welter(a_1, \ldots, a_k) = a_1 +_2 \ldots +_2 a_k +_2 \sum mate(a_i, a_j),$$

where all sums are nim sums, and the summation ranges over all pairs of distinct integers from the list of integers
(notice that the mating function is applied to each pair).

In [88]:
def welter(L):
    '''
    Input: A list of integers
    Output: The Welter function applied to the list
    '''
    val = 0
    k = len(L)
    for i in range(k):
        for j in range(i+1, k):
            val = xorE( val, mate(L[i],L[j]) )

    return xorE( nim(L), val )

def test_welter():
    print("welter([1,3,5,11]) = 4?", welter([1,3,5,11]) )
    print("welter([2,3,5,7,11,13,17,19,23]) = 28? (Conway example)", welter([2,3,5,7,11,13,17,19,23]) )
    print("welter([4,0,2,4]) = -2?", welter([4,0,2,4]) )

In [89]:
test_welter()

welter([1,3,5,11]) = 4? 4
welter([2,3,5,7,11,13,17,19,23]) = 28? (Conway example) 28
welter([4,0,2,4]) = -2? -2


## Inverting Welter Function

In order to play the game, we need to be able to change an entry in a list so that we can get any new
desired Welter value.  When negative integers are allowed this can always be done.

In [92]:
def Winversion(L, i, n, M):
    '''
    Input: list of integers L, index i into L, integer n, integer M (max search)
    NOTE: Negatives not done yet!!
    Output: searches (0 to M-1) for an integer at index i, so that the welter value is n
            (current value at index i is irrelevant)
            By the Bijective Fact, there soes exist such an integer (but we do not have bounds in its size) 
            returns False if not found.
    '''
    X = L.copy()
    for v in range(M):
        X[i] = v
        if welter(X) == n:
            return v

    return False

def test_Winversion():
    print("Winversion([1,3,5,11], 1, 15, 10) = 6?", Winversion([1,3,5,11], 1, 15, 10) )
    print("Winversion([1,3,5,11], 1, 15, 6) = False?", Winversion([1,3,5,11], 1, 15, 6) )
    

In [93]:
test_Winversion()

Winversion([1,3,5,11], 1, 15, 10) = 6? 6
Winversion([1,3,5,11], 1, 15, 6) = False? False


### Basic Experimenting and Understanding of Welter and Inverion

The position [1, 3, 5, 11] can be seen to have Welter value 4.  Suppose we want to know how to change each 
entry individually in order for its value to become 3.  For any new value, and any entry, there is always a way to achieve that value
since the Welter function is a bijection from Z to Z in any one of its arguments. Importantly for this to be true, we must allow for the extention to negative numbers.

In [100]:
print("Original Welter value:", welter([1,3,5,11] ) )

# Invert the 4 positions for the value of 3
print("To get value of 3.")
print( "Change position 0 to", Winversion([1,3,5,11], 0, 3, 100) )
print( "Change position 1 to", Winversion([1,3,5,11], 1, 3, 100) )
print( "Change position 2 to", Winversion([1,3,5,11], 2, 3, 100) )
print( "Change position 3 to", Winversion([1,3,5,11], 3, 3, 100) )

# Check above (good - all 3)
print()
print( "Check that above values work. Should all be 3." )
print( welter([0,3,5,11]) )
print( welter([1,10,5,11]) )
print( welter([1,3,4,11]) )
print( welter([1,3,5,2]) )

Original Welter value: 4
To get value of 3.
Change position 0 to 0
Change position 1 to 10
Change position 2 to 4
Change position 3 to 2

Check that above values work. Should all be 3.
3
3
3
3


## Experiments- evidence for ``Claim A'' (p. 159)

Claim A: $ [a_1 | a_2 | \cdots | a_k ] = 
[ [ a_1 | a_3 | a_4 | \cdots | a_k ] | [ a_2 | a_3 | a_4 | \cdots | a_k ] ]  
+_2 [ a_3 | a_4 | \cdots | a_k ] $

In [18]:
a1, a2, a3, a4, a5 = 2, 3, 5, 8, 9
print(welter([a1, a2, a3, a4, a5]))
L = welter([a1, a3, a4, a5])
R = welter([a2, a3, a4, a5])
X = nim([L, R]) - 1
print( nim( [X, welter([a3, a4, a5])] ) )


1
1


In [19]:
a1, a2, a3, a4, a5 = 2, 4, 9, 14, 20
print(welter([a1, a2, a3, a4, a5]))
L = welter([a1, a3, a4, a5])
R = welter([a2, a3, a4, a5])
X = nim([L, R]) - 1
print( nim( [X, welter([a3, a4, a5])] ) )

13
13


## Experiments- evidence for ``Claim B'' (p. 159)

Claim B: $( [a_1 | a_3 | a_4 | \cdots | a_k ] = [a_2 | a_3 | \cdots | a_k ] = (a_1 | a_2) $

In [21]:
a1, a2, a3, a4, a5 = 2, 3, 5, 8, 9
print( mate( welter([a1, a3, a4, a5]), welter([a2, a3, a4, a5]) ) )
print( mate(a1,a2) )

1
1


In [22]:
a1, a2, a3, a4, a5 = 2, 4, 9, 14, 20
print( mate( welter([a1, a3, a4, a5]), welter([a2, a3, a4, a5]) ) )
print( mate(a1,a2) )

3
3


## Even Alternation Theory

Conway develops what is called Even Alternation Theory.  To write the following:
$
\left[ \begin{array}{llll}
a_1 & a_2 & \cdots & a_k \\
a_1' & a_2' & \cdots & a_k' \\
\end{array}\right] 
= 
\left[\begin{array}{c}
n \\
n'\end{array}
\right],
$
means that Welter$([a_1, \ldots, a_k]) = n$ and furthermore, if any even number of the primed numbers are swapped up to the top row, we get another true equation.  This is illustrated in the computational example below, which builds on some above calculations.



In [110]:
print("Consider this the top row, [1, 3, 5, 11]  =", welter([1,3,5,11] ) )
print("Consider this the bot row, [0,10, 4, 11]) =", welter([0,10, 4, 11]) )
print()
print("Swap positions 3 and 4 up, and leave result of 4, so [1,3, 4, 2]) =", welter([1,3, 4, 2]) )
print("Swap bot positions 3 and result of 3 up ([1,3, 4, 11]) =", welter([1,3, 4, 11]) )


Consider this the top row, [1, 3, 5, 11]  = 4
Consider this the bot row, [0,10, 4, 11]) = 3

Swap positions 3 and 4 up, and leave result of 4, so [1,3, 4, 2]) = 4
Swap bot positions 3 and result of 3 up ([1,3, 4, 11]) = 3


In [117]:
# UNDER CONSTRUCTION!  The class is functional and used below, but there are some methods under construction

class AlternationArray:
    '''
    Data structure to represent the 2 rows in the even alternation theory of Welter
    '''
    def __init__(self):
        self.topRow = None
        self.topVal = None
        self.botRow = None
        self.botVal = None
        self.M = 0

    def setMaxSearch(self, M):
        self.M = M

    def setTopRow(self, L):
        self.topRow = L

    def setBotRow(self, L):
        self.botRow = L

    def setTopVal(self, v):
        self.topVal = v

    def setBotVal(self, v):
        self.botVal = v

    def getTopRow(self):
        return self.topRow

    def getBotRow(self):
        return self.botRow

    def getTopVal(self):
        return self.topVal

    def getBotVal(self):
        return self.botVal

    def calcTopVal(self):
        self.topVal = welter(self.topRow) 

    def calcBotRow(self):
        'Assumes top row and both values set.  Calculates and sets bottom row'
        k = len(self.topRow)
        self.botRow = [0]*k
        for i in range(k):
            self.botRow[i] = Winversion(self.topRow, i, self.botVal, self.M)

    def doAllSwaps(self, num):
        # NOTE: NOT COMPLETED.
        '''
        Does all of num swaps up, indicating the result and its True/False.
        Returns as list of triples where each triple (L, v, b) is:
        - L is the swapped up list values
        - v is the value it is supposed to be (original or swapped up)
        - b is True if welter(L) = v and False otherwise
        '''
        countL = list(range(len(self.botRow)))
        result = []

        # Swaps where top value is NOT changed
        for S in list(it.combinations(countL,num)):
            newRow = self.topRow.copy()
            for i in S:
                newRow[i] = self.botRow[i]
            newVal = welter(newRow)
            truth  = (newVal == self.topVal)
            result.append( [newRow, newVal, truth] )

        # Swaps where top value is IS changed     
        # PUT CODE HERE ...

        return result

    def isUnusual(self):
        '''
        Returns True if it has some unusual trait currently checked for:
        - any repeat among top or bottom row
        '''
        L = self.topRow + self.botRow
        if len(L) != len(set(L)):
            return True

        return False
        

    def __str__(self):
        S =  str(self.topRow) + ' | ' + str(self.topVal) + '\n'
        S += str(self.botRow) + ' | ' + str(self.botVal)
                
        return S

In [169]:
# NOT MATCH THEORY!?
# Have bottom row not distinct from top, and thus problem in getting a legal move.

EAT6 = AlternationArray()
EAT6.setMaxSearch(100)

EAT6.setTopRow([0, 1, 2, 4, 6])
EAT6.calcTopVal()
EAT6.setBotVal(0)
EAT6.calcBotRow()
print(EAT6)

print()
print("Checking inversion from scratch ..")
print( Winversion([0, 1, 2, 4, 6], 0, 0, 100) )
print( Winversion([0, 1, 2, 4, 6], 1, 0, 100) )
print( Winversion([0, 1, 2, 4, 6], 2, 0, 100) )
print( Winversion([0, 1, 2, 4, 6], 3, 0, 100) )
print( Winversion([0, 1, 2, 4, 6], 4, 0, 100) )

print()
print("Checking some even swaps up ..")
print("Is it 0?:", welter([9, 1, 2, 4, 6]) )
print("Is it 0? (no since not all distinct?):", welter([0, 6, 2, 4, 6]) )
print("Is it 0?:", welter([0, 1, 7, 4, 6]) )

[0, 1, 2, 4, 6] | 1
[9, 8, 7, 5, 3] | 0

Checking inversion from scratch ..
9
8
7
5
3

Checking some even swaps up ..
Is it 0?: 0
Is it 0? (no since not all distinct?): -2
Is it 0?: 0


### Experiment with 4 coin game

In [26]:
EAT = AlternationArray()
EAT.setMaxSearch(100)
EAT.setTopRow([1,3,5,11])
EAT.calcTopVal()

# bottom value = 3
EAT.setBotVal(3)
EAT.calcBotRow()
print(EAT)
print()

# bottom value = 2
EAT.setBotVal(2)
EAT.calcBotRow()
print(EAT)
print()

[1, 3, 5, 11] | 4
[0, 10, 4, 2] | 3

[1, 3, 5, 11] | 4
[19, 17, 7, 9] | 2



In [27]:
# bottom value = 1
EAT.setBotVal(1)
EAT.calcBotRow()
print(EAT)
print()

[1, 3, 5, 11] | 4
[2, 8, 6, 0] | 1



In [28]:
# bottom value = 0
EAT.setBotVal(0)
EAT.calcBotRow()
print(EAT)
print()

[1, 3, 5, 11] | 4
[13, 15, 9, 7] | 0



### Experiment with 5 coin game

In [29]:
EAT5 = AlternationArray()
EAT5.setMaxSearch(100)
EAT5.setTopRow([2,4,5,9, 12])
EAT5.calcTopVal()

In [30]:
# bottom value = 13
EAT5.setBotVal(13)
EAT5.calcBotRow()
print(EAT5)
print()

# bottom value = 0
EAT5.setBotVal(0)
EAT5.calcBotRow()
print(EAT5)
print()

[2, 4, 5, 9, 12] | 14
[25, 11, 6, 18, 3] | 13

[2, 4, 5, 9, 12] | 14
[28, 10, 15, 3, 18] | 0



### More systematic Evidence for Theorem 80 in Conway

Ran the below code with the below parameters, which runs through 1000s of examples.  
It is evidence that Theorem 80
from Conway is actually true (with the added condition that in fact
all the numerical values are distinct ... as desired!)

- k = 3, m = 15, b = 5
- k = 3, m = 20, b = 9
- k = 4, m = 15, b = 5
- k = 5, m = 15, b = 4
- k = 6, m = 12, b = 2

In [31]:
# Search for counter examples

EATr = AlternationArray()
EATr.setMaxSearch(100)

k = 6 # length of rows, i.e. number of coins
m = 12 # max coin position
b = 2 # bottom values range 0 to b, inclusive

countL = list(range(m+1))

unusual = False
for S in list(it.combinations(countL,k)):
    EATr.setTopRow(list(S))
    EATr.calcTopVal()
    for x in range(b+1):
        if x != EATr.getTopVal():
            EATr.setBotVal(x)
            EATr.calcBotRow()

            if EATr.isUnusual():
                print("UNUSUAL EAT!!")
                print(EATr)
                print()

print()
if unusual == False:
    print("NOTHING UNUSUAL")


NOTHING UNUSUAL


## Play Game!

Below is a class allowing the playing of the game.

In [115]:
# The function findDecrease is a basic utility function used in the class below

def findDecrease(A, B):
    '''
    Given 2 lists of integers A and B, finds first index where B is smaller
    Returns None if it fails.
    '''
    for i in range(len(A)):
        if A[i] > B[i]:
            return i

    return None

def test_findDecrease():
    A = [4, 5, 3, 8]
    B = [6, 9, 1, 4]
    print("Decrease at index 2?", findDecrease(A, B) )

    A = [4, 5, 3, 8]
    B = [6, 9, 3, 55]
    print("No Decrease?", findDecrease(A, B) )

In [116]:
test_findDecrease()

Decrease at index 2? 2
No Decrease? None


In [160]:
gameRules = "SUMMARY OF RULES: \
There will be some number of coins on a horizontal strip, in numbered slots starting at 0. \
On your turn you move a coin from its slot to a lower numbered slot (remember: jumping another coin IS allowed). \
The first player who cannot move loses. \
In this console interface a coin position will be displayed as for example: \
\n      0 * * 3 4 * 6\n\
where the slots with coins are indicated by the numbers, and those that are empty are indicated by a '*'. \
On your turn select one of the filled slots (e.g. 0, 3, 4, or 6), then indicate which slot you want to move it to. \
For example the coin at slot 3 could be moved to slot 1 or 2.  The coin at slot 0 can never be moved."

In [170]:
class WelterGame:
    'Class for playing the Welter game'
    def __init__(self):
        self.startCoin = 2 # max position of start coin
        self.incrCoin  = 3 # max increment to next coin
        self.TEMP = 2 # set bottom value when not 0

        self.EAT = AlternationArray()
        self.EAT.setMaxSearch(100)
        self.board = None

    def play(self):
        print(gameRules)
        print()
        self.numCoin = int(input("How many coins do you want to play with: "))
        self.board = self.createBoard()

        print("Starting board: ", end = '')
        self.displayBoard()
        print()
        while True:
            try:
                turn = int(input("Do you want to go first (1) or second (2): "))
            except:
                print("Yo! Give me an integer ...")

            if turn == 1 or turn == 2:
                break

            print("Need to give me a 1 or a 2, please ...")
        print()
            
        if turn == 1:
            turn = 'PLAYER'
        else:
            turn = 'COMPUTER'

        # game loop
        while self.isOver() == False:
            if turn == 'PLAYER':
                playVal = self.playerMove()
                if playVal == False:
                    break
            else:
                self.computerMove()

            if turn == 'PLAYER':
                print("Board after your move: ", end = '')
            else:
                print("Board after my move: ", end = '')
            self.displayBoard()
            print()
            print()

            # swap turns
            if turn == 'PLAYER':
                turn = 'COMPUTER'
            else:
                turn = 'PLAYER'

        print()
        if turn == 'COMPUTER': # i.e. the computer could not move on its turn
            print("You won!")
        else:
            print("I won :-) ...")
        print("*** GAME OVER ***")
        print()

    
    def playerMove(self):

        # Player query loop
        while True: 
            toMove = input("Piece to move (Q to Quit): ")
            if toMove.lower() == 'q':
                return False

            keepGoing = True
            try:
                toMove = int(toMove)
            except:
                print("Please give me the integer representing some coin.")
                keepGoing = False

            if keepGoing:
                try:
                    whereToMove = int(input("Where to move: "))
                except:
                    print("Please give me the integer representing an open position.")
                    keepGoing = False

                if keepGoing:
                    if toMove not in self.board:
                        print("Need to give me the integer of an actual coin")
                    elif whereToMove in self.board or whereToMove >= toMove or whereToMove < 0:
                        print("Illegal move!")
                    else:
                        break

            
        
        self.board[ self.board.index(toMove) ] = whereToMove
        self.board.sort()

        return True
    
    def computerMove(self):
        self.EAT.setTopRow(self.board)
        self.EAT.calcTopVal()
        
        if self.EAT.getTopVal() != 0:
            self.EAT.setBotVal(0)
        else:
            self.EAT.setBotVal(self.TEMP)
            
        self.EAT.calcBotRow()
        index = findDecrease(self.EAT.getTopRow(), self.EAT.getBotRow())
        
        if index == None: # no decrease since no winning move
            self.randomMove()
        else: # else have a legal move
            self.board[index] = self.EAT.getBotRow()[index]
            
        self.board.sort()

    def randomMove(self):
        v = self.board[-1]
        for i in range(v):
            if i not in self.board:
                self.board[-1] = i
                break
    
    def createBoard(self):
        L = []
        L.append( rd.randint(0, self.startCoin) )

        for i in range(self.numCoin - 1):
            L.append( L[-1] + rd.randint(1,self.incrCoin) )
        
        return L

    
    def isOver(self):
        'Over and returns True if the board is [0,1,2,...]'
        for i in range(len(self.board)):
            if self.board[i] != i:
                return False
                
        return True

    def displayBoard(self):
        for x in range(self.board[-1] + 1): # go up to and including the last and largest element on the board.
            if x in self.board:
                print(f" {x} ", end = '')
            else:
                print(" * ", end = '')

In [171]:
# TEST some of WelterGame
testG = WelterGame()
testG.board = [1, 3, 4, 7, 8]
print("Start board value:", welter(testG.board) ) 
testG.computerMove()
print("End board value:", welter(testG.board) ) 
print("End board:", testG.board)
testG.displayBoard()

Start board value: 9
End board value: 0
End board: [0, 3, 4, 7, 8]
 0  *  *  3  4  *  *  7  8 

### To play the game, run the following cell

In [172]:
WG = WelterGame()
WG.play()

SUMMARY OF RULES: There will be some number of coins on a horizontal strip, in numbered slots starting at 0. On your turn you move a coin from its slot to a lower numbered slot (remember: jumping another coin IS allowed). The first player who cannot move loses. In this console interface a coin position will be displayed as for example: 
      0 * * 3 4 * 6
where the slots with coins are indicated by the numbers, and those that are empty are indicated by a '*'. On your turn select one of the filled slots (e.g. 0, 3, 4, or 6), then indicate which slot you want to move it to. For example the coin at slot 3 could be moved to slot 1 or 2.  The coin at slot 0 can never be moved.



How many coins do you want to play with:  5


Starting board:  *  1  *  *  4  *  6  *  8  *  10 


Do you want to go first (1) or second (2):  2



Board after my move:  *  1  *  3  4  *  *  *  8  *  10 



...

Board after your move:  0  1  *  3  *  *  6  7 

Board after my move:  0  1  *  3  *  5  *  7 



Piece to move (Q to Quit):  5
Where to move:  6


Illegal move!


Piece to move (Q to Quit):  5
Where to move:  4


Board after your move:  0  1  *  3  4  *  *  7 

Board after my move:  0  1  2  3  4 


I won :-) ...
*** GAME OVER ***



## To do??

- Could write a more efficient Welter function which calculates the value using maximum mated pairs (though this approach does not work for all cases - does work on game-like positions)
- Could write a function which turns some Welter list $[a_1, \ldots, a_k]$ into an animating function from Z to Z by making one argument vary.  Then can see evidence that this resulting animating function is a bijection.
- Can add to the Even Alternation programs, a check that there is an index where there is a smaller value at the bottom row than the top row (if all the values are distinct and positive)
- Could do experiments to give evidence for Swap Lemma.